In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [2]:
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.tools.wikipedia.tool import WikipediaAPIWrapper

C:\Users\kanan\AppData\Local\Temp\ipykernel_516\3438664346.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.wikipedia.tool import WikipediaQueryRun


In [3]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [4]:

wiki_tool,type(wiki_tool), wiki_tool.name

(WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\work\\langchain\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 langchain_community.tools.wikipedia.tool.WikipediaQueryRun,
 'wikipedia')

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://reference.langchain.com/python/langchain-community/utilities/wikipedia/WikipediaAPIWrapper")
docs = loader.load()

document = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
faiss_db = FAISS.from_documents(document,GoogleGenerativeAIEmbeddings(model="gemini-embedding-2"))
retriever = faiss_db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001BAC9E589D0>, search_kwargs={})

In [6]:
#create retriever tool
from langchain_core.tools.retriever import create_retriever_tool

faiss_retrieval_tool = create_retriever_tool(retriever,"langchain_wiki","Give information about WkikpeidaAAIWrapper and its methods")

In [7]:
faiss_retrieval_tool.name

'langchain_wiki'

In [8]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)

arxiv.name

'arxiv'

In [9]:
#combine all 3 (wiki_tool, faiss_retrieval_tool,arxiv)
tools = [wiki_tool,faiss_retrieval_tool,arxiv]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\work\\langchain\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langchain_wiki', description='Give information about WkikpeidaAAIWrapper and its methods', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001BAAD9AEFC0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001BAC9D96B60>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200))]

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash')


In [11]:
#using langchain hub to get precreated prompt templete instead of usign createChatPromptTemplete

from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

client = Client()
pull_prompt = client.pull_prompt("homanp/question-answer-pair",dangerously_pull_public_prompt=True)
#prompt.messages.append(MessagesPlaceholder(variable_name="agent_scratchpad"))
type(pull_prompt)
prompt = ChatPromptTemplate.from_messages([
    *pull_prompt.messages,
    MessagesPlaceholder(variable_name="agent_scratchpad")
])


In [19]:
# Create a LangSmith API in Settings > API Keys
# Make sure API key env var is set:
# import os; os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"
from langsmith import Client

client = Client()
prompt2 = client.pull_prompt("hwchase17/openai-functions-agent",dangerously_pull_public_prompt=True)
prompt2

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='

In [17]:
#query from the tools using agents

from langchain_classic.agents import create_tool_calling_agent

agent = create_tool_calling_agent(llm,tools,prompt2)


In [18]:
#using agent executer
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True)

In [20]:
#invoking the agent_executor
agent_executor.invoke({"input":"what is Machine Learning?"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from pre-trained [{'type': 'text', 'text': '**Machine Learning (ML)** is a branch of artificial intelligence (AI) and computer science that focuses on training computers to learn from data and make decisions or predictions without being explicitly programmed to do so. \n\nInstead of writing specific code with a fixed set of rules to solve a problem, you feed a machine learning algorithm a large amount of data, and the algorithm "learns" the underlying patterns to figure out how to solve the problem on its own.\n\n---\n\n### How it Differs from Traditional Programming\n\n* **Traditional Programming:** You write the **Rules** (code) and input the **Data**, and the computer outputs the **Answers**.\n  * *Exam

{'input': 'what is Machine Learning?',
 'output': [{'type': 'text',
   'text': '**Machine Learning (ML)** is a branch of artificial intelligence (AI) and computer science that focuses on training computers to learn from data and make decisions or predictions without being explicitly programmed to do so. \n\nInstead of writing specific code with a fixed set of rules to solve a problem, you feed a machine learning algorithm a large amount of data, and the algorithm "learns" the underlying patterns to figure out how to solve the problem on its own.\n\n---\n\n### How it Differs from Traditional Programming\n\n* **Traditional Programming:** You write the **Rules** (code) and input the **Data**, and the computer outputs the **Answers**.\n  * *Example:* If temperature > 30°C, turn on AC.\n* **Machine Learning:** You input the **Data** and the expected **Answers** (or let it find patterns), and the computer generates the **Rules** (called a *model*).\n  * *Example:* You feed the computer thous

In [21]:
agent_executor.invoke({"input":"what are the methods of WikipediaAPIWrapper?"})



> Entering new AgentExecutor chain...

Invoking: `langchain_wiki` with `{'query': 'methods of WikipediaAPIWrapper'}`


Anyattributetop_k_results: intattributelang: strattributeload_all_available_meta: boolattributedoc_content_chars_max: intmethodvalidate_environmentValidate that the python package exists in environment.methodrunRun Wikipedia search and get page summaries.methodloadRun Wikipedia search and get the article text plus the meta information.

lineMenuNavigationProjectsAmazon NovaAnthropicAstraDBAWSAzure (Microsoft)CerebrasChromaCohereCommunityOverviewChat ModelsLLMsEmbeddingsVector StoresDocument LoadersTools & AgentsRetrieversCachesChat HistoryGraphsUtilitiesDb2DeepSeekElasticsearchExaFireworksGoogle (Community)Google GenAI (Gemini)Google Vertex AIGroqHuggingFaceIBMLiteLLMMilvusMistral AIMongoDBNeo4JNomicNvidia AI EndpointsOllamaOpenAIOpenRouterParallelPerplexityPineconePostgresQdrantRedisSema4SnowflakeSQLServerTavilyTogetherUnstructuredUpstageWeaviatexAILanguagePythonJav

{'input': 'what are the methods of WikipediaAPIWrapper?',
 'output': [{'type': 'text',
   'text': "The `WikipediaAPIWrapper` class in LangChain (specifically from the `langchain_community.utilities` module) has several key methods used to interact with the Wikipedia API:\n\n### 1. **`run(query: str) -> str`**\n* **Description:** Runs a Wikipedia search for the given query and returns a single concatenated string containing the page summaries of the top results (limited by `top_k_results`).\n* **Usage:** Best for quickly getting a summary text of a topic.\n\n### 2. **`load(query: str) -> List[Document]`**\n* **Description:** Runs a Wikipedia search and retrieves the full article text along with metadata (such as the source URL, title, and publish date) for the top results. It returns this information as a list of LangChain `Document` objects.\n* **Usage:** Best when you need structured document objects containing both the page content and metadata for downstream tasks (e.g., loading int